In [14]:
!conda run -p /home/njm12/ATMS_523/envs/xarray-climate python -m ipykernel install --user --name xarray-climate --display-name "Python (xarray-climate)"

Installed kernelspec xarray-climate in /home/njm12/.local/share/jupyter/kernels/xarray-climate



In [15]:
import sys
print(sys.executable)

/home/njm12/ATMS_523/envs/xarray-climate/bin/python


In [16]:
# ================================================================
# Imports
# ================================================================

import pandas as pd
from astral.sun import sun
from astral import LocationInfo
import pytz

In [17]:
# ================================================================
# File paths
# ================================================================

tornado_csv = "/home/njm12/ATMS_596/1950-2024_actual_tornadoes.csv"
output_file = "/home/njm12/ATMS_596/ATMS-596-Capstone-Project/CSV Files/tornadoes_with_solar_times_and_nocturnal_bins.csv"

# ================================================================
# Configuration
# ================================================================

states_of_interest = ["MO", "IA", "IL"]
min_date = pd.to_datetime("1950-01-01")


In [18]:
# ================================================================
# Load data
# ================================================================

tornadoes = pd.read_csv(tornado_csv, parse_dates=["date"])

# Filter
tornadoes = tornadoes[
    tornadoes["st"].isin(states_of_interest) &
    tornadoes["mag"].isin([0,1,2,3,4,5]) &
    (tornadoes["date"] >= min_date)
].copy()

print("Filtered tornadoes:", tornadoes.shape)

# ================================================================
# Fix longitude + create datetime
# ================================================================

tornadoes["slon"] = tornadoes["slon"].apply(lambda x: -x if x > 0 else x)

tornadoes["datetime"] = pd.to_datetime(
    tornadoes["date"].astype(str) + " " + tornadoes["time"].astype(str)
)

central = pytz.timezone("America/Chicago")
tornadoes["datetime"] = tornadoes["datetime"].dt.tz_localize(central)

Filtered tornadoes: (8287, 29)


In [19]:
# ================================================================
# Compute solar times
# ================================================================

def compute_solar_times(row):

    lat = row["slat"]
    lon = -abs(row["slon"])
    date = row["date"]

    try:
        location = LocationInfo(latitude=lat, longitude=lon)
        s = sun(location.observer, date=date, tzinfo=central)

        sunrise = s["sunrise"].astimezone(central)
        sunset = s["sunset"].astimezone(central)

        # Ensure sunset is after sunrise
        if sunset < sunrise:
            sunset += pd.Timedelta(days=1)

        return pd.Series({
            "sunrise_local": sunrise,
            "sunset_local": sunset
        })

    except:
        return pd.Series({
            "sunrise_local": pd.NaT,
            "sunset_local": pd.NaT
        })

solar_times = tornadoes.apply(compute_solar_times, axis=1)
df = pd.concat([tornadoes, solar_times], axis=1)

In [20]:
# ================================================================
# Day vs Night classification
# ================================================================

def classify_day_night(row):

    event_time = row["datetime"]
    sunrise = row["sunrise_local"]
    sunset = row["sunset_local"]

    if pd.isnull(sunrise) or pd.isnull(sunset):
        return None

    if sunrise <= event_time <= sunset:
        return "Day"
    else:
        return "Night"

df["day_night_bin"] = df.apply(classify_day_night, axis=1)

In [21]:
# ================================================================
# Repair any missing classifications
# ================================================================

missing = df[df["day_night_bin"].isna()].copy()
print("Rows missing solar classification:", len(missing))

for idx, row in missing.iterrows():

    try:
        lat = row["slat"]
        lon = -abs(row["slon"])
        date = row["date"]

        location = LocationInfo(latitude=lat, longitude=lon)
        s = sun(location.observer, date=date, tzinfo=central)

        sunrise = s["sunrise"].astimezone(central)
        sunset = s["sunset"].astimezone(central)

        if sunset < sunrise:
            sunset += pd.Timedelta(days=1)

        df.loc[idx, "sunrise_local"] = sunrise
        df.loc[idx, "sunset_local"] = sunset

        event_time = df.loc[idx, "datetime"]

        if sunrise <= event_time <= sunset:
            df.loc[idx, "day_night_bin"] = "Day"
        else:
            df.loc[idx, "day_night_bin"] = "Night"

    except Exception as e:
        print("Still failed:", idx, e)

print("Remaining missing:", df["day_night_bin"].isna().sum())

Rows missing solar classification: 0
Remaining missing: 0


In [22]:
# ================================================================
# Refined nighttime classification
# ================================================================

def classify_night_subtypes(row):
    
    event_time = row["datetime"]
    sunrise = row["sunrise_local"]
    sunset = row["sunset_local"]
    
    if pd.isnull(sunrise) or pd.isnull(sunset):
        return None
    
    # Day remains unchanged
    if row["day_night_bin"] == "Day":
        return "Day"
    
    evening_end = sunset + pd.Timedelta(hours=3)
    morning_start = sunrise - pd.Timedelta(hours=3)
    
    # Evening transition
    if sunset <= event_time <= evening_end:
        return "Evening Transition"
    
    # Morning transition
    elif morning_start <= event_time <= sunrise:
        return "Morning Transition"
    
    # Core night
    else:
        return "Core Night"

df["night_subtype"] = df.apply(classify_night_subtypes, axis=1)

In [23]:
# ================================================================
# Day/night duration diagnostics
# ================================================================

df["diurnal_sec"] = (df["sunset_local"] - df["sunrise_local"]).dt.total_seconds()
df["nocturnal_sec"] = 86400 - df["diurnal_sec"]

# Convert to hours

df["diurnal_hours"] = df["diurnal_sec"] / 3600
df["nocturnal_hours"] = df["nocturnal_sec"] / 3600

In [24]:
# ================================================================
# Compute AVERAGE Core Night Duration ONLY (FIXED)
# ================================================================

df["evening_end"] = df["sunset_local"] + pd.Timedelta(hours=3)

# Shift sunrise to NEXT day
df["next_sunrise"] = df["sunrise_local"] + pd.Timedelta(days=1)

df["morning_start"] = df["next_sunrise"] - pd.Timedelta(hours=3)

df["core_night_hours"] = (
    (df["morning_start"] - df["evening_end"]).dt.total_seconds() / 3600
)

print("Average Core Night Duration (hours):", df["core_night_hours"].mean())

Average Core Night Duration (hours): 4.42617786118881


In [25]:
# ================================================================
# Sanity checks
# ================================================================

print("\nDay/Night counts:")
print(df["day_night_bin"].value_counts())

print("\nNight subtype counts:")
print(df["night_subtype"].value_counts())

print("\nMin daylight hours:", df["diurnal_hours"].min())
print("Max daylight hours:", df["diurnal_hours"].max())


Day/Night counts:
day_night_bin
Day      6155
Night    2132
Name: count, dtype: int64

Night subtype counts:
night_subtype
Day                   6155
Evening Transition    1222
Core Night             712
Morning Transition     198
Name: count, dtype: int64

Min daylight hours: 8.960105314722222
Max daylight hours: 15.412055629166666


In [26]:
# ================================================================
# Save output
# ================================================================

df.to_csv(output_file, index=False)
print("Saved to:", output_file)

Saved to: /home/njm12/ATMS_596/ATMS-596-Capstone-Project/CSV Files/tornadoes_with_solar_times_and_nocturnal_bins.csv
